In [1]:
# This notebook is used to process the raw data from the ENFSI database so that they can be read by the neural network training scripts. 
#It is also used to create training and test datasets for each functional group under investigation.
#
# Briefly, for each molecule in the database, the JDX files are converted into CSV files containing the spectrum within the appropriate spectral range 
#and at the required resolution. Based on the InChI identifiers included in the JDX file contents, the generated CSV files are saved in directories 
#corresponding to the respective functional groups.
#
# Subsequently, spectra of molecules that contain and do not contain the functional group under investigation are randomly selected to create 
#a balanced training and test dataset.

# Note, that this script is for ENFSI dataset only, script for data preparation for NIST dataset is available at https://github.com/marek-krosnicki/FunctionalGrups/

In [2]:
#path to directory with .jdx files downoladed from ENFSI webpage
jdx_file_path='C:\\Users\\Tomek_\\Desktop\\cc\\ENFSI\\'

#directory for all .csv files
save_dirpath='C:\\Users\\Tomek_\\Desktop\\cc\\prepENFSI1\\'

#directories for files associated with analysed functional groups (subfolders)
FGPath='C:\\Users\\Tomek_\\Desktop\\cc\\FGYES1\\'
FGPathNOT='C:\\Users\\Tomek_\\Desktop\\cc\\FGNOT1\\'
FGBoth='C:\\Users\\Tomek_\\Desktop\\cc\\FG1\\'

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
import numpy as np
import os
import pandas as pd 
import numpy as np

from rdkit import Chem, RDLogger

#definition of analysed functional groups

func_grp_smarts = {'alkane':'[CX4;H0,H1,H2,H4]',
                   'methyl':'[CH3]',
                   'alkene':'[CX3]=[CX3]',
                   'alkyne':'[CX2]#C',
                   'alcohols':'[#6][OX2H]',
                   'amines':'[NX3;H2,H1;!$(NC=O)]', 
                   'nitriles':'[NX1]#[CX2]', 
                   'aromatics':'[$([cX3](:*):*),$([cX2+](:*):*)]',
                   'alkyl halides':'[#6][F,Cl,Br,I]', 
                   'esters':'[#6][CX3](=O)[OX2H0][#6]', 
                   'ketones':'[#6][CX3](=O)[#6]',
                   'aldehydes':'[CX3H1](=O)[#6]', 
                   'carboxylic acids':'[CX3](=O)[OX2H1]', 
                   'ether': '[OD2]([#6])[#6]',
                   'acyl halides':'[CX3](=[OX1])[F,Cl,Br,I]',
                   'amides':'[NX3][CX3](=[OX1])[#6]',
                   'nitro':'[$([NX3](=O)=O),$([NX3+](=O)[O-])][!#8]',
                   'heterocyclic': '[!#6;!R0]',
                   'aryl chlorides': '[Cl][c]',
                   'carboxylic esters': '[CX3;$([R0][#6]),$([H1R0])](=[OX1])[OX2][#6;!$(C=[O,N,S])]',
                   'alkyl aryl ethers': '[OX2](c)[CX4;!$(C([OX2])[O,S,#7,#15,F,Cl,Br,I])]',
                   'phenols': '[OX2H][c]',
                   'trifluoromethyl':'[#6](F)(F)F',
                   'isocyanate':'[*][N]=C=O',
                   'phosphate':'OP(=O)(O)O' ,
                   'fluorinated':'[CX4,cX3][F]'}



func_grp_structs = {func_name : Chem.MolFromSmarts(func_smarts)\
                        for func_name, func_smarts in func_grp_smarts.items()}

def normalizeData(data: np.ndarray):
    return (data - data.min())/(data.max()-data.min())

def identify_functional_groups(inchi):  
    try:
        #Convert inchi to molecule
        mol = Chem.MolFromInchi(inchi, treatWarningAsError=True)   
        mol_func_grps = []

        #populate the list with binary values
        for _, func_struct in func_grp_structs.items():
            struct_matches = mol.GetSubstructMatches(func_struct)
            contains_func_grp = int(len(struct_matches)>0)
            mol_func_grps.append(contains_func_grp)
        return mol_func_grps
    except:
        return None



In [4]:

def specConverter(jdx_file_path,jdx_file_name):
    with open(jdx_file_path+jdx_file_name, 'r') as jdxfile:
            #all_counter += 1
            jdxfile = jdxfile.read().split('\n')
            
            for index, line in enumerate(jdxfile):
                    
                if line.startswith('##XUNITS'):
                    if line.split('##XUNITS=')[1] == 'MICROMETERS':
                        um = True

                    else:
                        um = False
                elif line.startswith('##CROSS REFERNCE'):
                    lines=line.split('; ');
                    inchiCode=lines[1].strip();
                    


                elif line.startswith('##YUNITS'):
                    #Get spectra type (transmittance/absorbance)
                    spectraType = line.strip("##").split('=')[1]
                    if line.startswith('##YUNITS=TRANSMITTANCE'):
                        transm = True 

                    else:
                        transm = False


                elif line.startswith('##FIRSTX'):
                    #Check first x in the spectra
                    firstX = float(line.strip('##').split('=')[1])


                elif line.startswith('##LASTX'):
                    #Check last x in the spectra
                    lastX = float(line.strip('##').split('=')[1])
                    if um:
                        firstX = umToInverseCm(firstX)
                        lastX = umToInverseCm(lastX)
            
                    reverse = False

                    if lastX < firstX:
                        reverse = True
                        cp = lastX
                        lastX = firstX
                        firstX = cp
                    

                    if firstX > 670:
                        break  
                    
                    if lastX < 3775:                 
                        break

                    #passed_counter += 1
                    

                elif line.startswith('##XYDATA'):
                    #Get spectra
                    spectra = jdxfile[index+1:-3]

                    try:
                        x, y = [line.split(' ')[0] for line in spectra], [line.split(' ')[1] for line in spectra]
                    except:
                        print(f'Corrupted file is: {jdx_file_path}')
                        break
                    if um:
                        x = umToInverseCmList(x)
                    
                    if reverse == True:
                        x.reverse()
                        y.reverse()

                        
                    y = normalizeData(CubicSpline(x, y)(np.arange(670, 3776, 1)))
                    if transm == True:
                        y = 1 - y                        
                        spectraType = "Converted_to_absorbance"
                        
                    #print(spectraType)

                    
                    
                    try:
                        funcG=identify_functional_groups(inchiCode);
                        #print(inchiCode)
                        #print(funcG)
                        funcG_ext= funcG+ [np.nan for _ in range(len(y)-len(funcG))]
                        #df = pd.DataFrame([y, 
                        #                    pd.concat(
                        #                        [identified_func_groups.loc[:, jdx_file_path.split('-IR.jdx')[0]],
                        #                        pd.Series(
                        #                            [np.nan for _ in range(len(y)-len(identified_func_groups.loc[:, jdx_file_path.split('-IR.jdx')[0]]))])
                        #                        ]
                        #                        ), 
                        #                        [spectraType] + [np.nan for _ in range(len(y)-1)]
                        #                    ], 
                        #                    index = ['y', 'funcGroups', 'spectraType']
                        #                ).T
                        df = pd.DataFrame([y, funcG_ext, [spectraType] + [np.nan for _ in range(len(y)-1)] ],index = ['y', 'funcGroups','spectraType']                                       
                                        ).T
                        sp=f"{os.path.join(save_dirpath, jdx_file_name.split('-IR.jdx')[0])}.csv"
                        df.to_csv(sp, index=False)

                        #print(sp)
                    except Exception as e:
                        print('error',e,jdx_file_name)
                        break
                    #print(len(y))
                    #print(len(funcG))
                    #swww=[spectraType] + [np.nan for _ in range(len(y)-1)]
                    #print(len(swww))
                    break

In [5]:
iter=0;
for file in os.listdir(jdx_file_path):
    iter+=1;
    #uncoment below, if You want generate only e.g. 100 files (for test/debuging)
    #if iter>100:
    #    break
    specConverter(jdx_file_path,file)

In [6]:
# Counting the number of molecules for each functional group
keys=list(func_grp_smarts.keys())
print(keys)

numberOfSpectra=[];
for i in range(len(keys)):
    numberOfSpectra.append(0)

for file in os.listdir(save_dirpath):
    df = pd.read_csv(save_dirpath+file)
    for i in range(len(keys)):
        if df['funcGroups'][i]==1:
            numberOfSpectra[i]=numberOfSpectra[i]+1;        
print(numberOfSpectra)





['alkane', 'methyl', 'alkene', 'alkyne', 'alcohols', 'amines', 'nitriles', 'aromatics', 'alkyl halides', 'esters', 'ketones', 'aldehydes', 'carboxylic acids', 'ether', 'acyl halides', 'amides', 'nitro', 'heterocyclic', 'aryl chlorides', 'carboxylic esters', 'alkyl aryl ethers', 'phenols', 'trifluoromethyl', 'isocyanate', 'phosphate', 'fluorinated']
[157, 142, 18, 0, 26, 90, 0, 150, 32, 16, 57, 1, 3, 74, 0, 7, 1, 94, 14, 16, 39, 4, 1, 0, 1, 14]


In [7]:
# We now calculate the size of the training-test dataset.
# NOTE: The dataset must be balanced. Its size is determined by the number of spectra
# containing and not containing the given functional group. We use twice the smaller
# of these two numbers.

iter=0;
for k in keys:
    howMuch=min(numberOfSpectra[iter],3854-numberOfSpectra[iter])*2
    print(k,";",howMuch)
    iter+=1

alkane ; 314
methyl ; 284
alkene ; 36
alkyne ; 0
alcohols ; 52
amines ; 180
nitriles ; 0
aromatics ; 300
alkyl halides ; 64
esters ; 32
ketones ; 114
aldehydes ; 2
carboxylic acids ; 6
ether ; 148
acyl halides ; 0
amides ; 14
nitro ; 2
heterocyclic ; 188
aryl chlorides ; 28
carboxylic esters ; 32
alkyl aryl ethers ; 78
phenols ; 8
trifluoromethyl ; 2
isocyanate ; 0
phosphate ; 2
fluorinated ; 28


In [8]:
#preparation of directories for training-test datasets for all analysed functional groups


for i in keys:
    #print(i)   
    xp=FGPath+i
    xp1=FGPathNOT+i
    xp2=FGBoth+i
    print(xp)
    os.makedirs(xp, exist_ok=True)
    os.makedirs(xp1, exist_ok=True)
    os.makedirs(xp2, exist_ok=True)

C:\Users\Tomek_\Desktop\cc\FGYES1\alkane
C:\Users\Tomek_\Desktop\cc\FGYES1\methyl
C:\Users\Tomek_\Desktop\cc\FGYES1\alkene
C:\Users\Tomek_\Desktop\cc\FGYES1\alkyne
C:\Users\Tomek_\Desktop\cc\FGYES1\alcohols
C:\Users\Tomek_\Desktop\cc\FGYES1\amines
C:\Users\Tomek_\Desktop\cc\FGYES1\nitriles
C:\Users\Tomek_\Desktop\cc\FGYES1\aromatics
C:\Users\Tomek_\Desktop\cc\FGYES1\alkyl halides
C:\Users\Tomek_\Desktop\cc\FGYES1\esters
C:\Users\Tomek_\Desktop\cc\FGYES1\ketones
C:\Users\Tomek_\Desktop\cc\FGYES1\aldehydes
C:\Users\Tomek_\Desktop\cc\FGYES1\carboxylic acids
C:\Users\Tomek_\Desktop\cc\FGYES1\ether
C:\Users\Tomek_\Desktop\cc\FGYES1\acyl halides
C:\Users\Tomek_\Desktop\cc\FGYES1\amides
C:\Users\Tomek_\Desktop\cc\FGYES1\nitro
C:\Users\Tomek_\Desktop\cc\FGYES1\heterocyclic
C:\Users\Tomek_\Desktop\cc\FGYES1\aryl chlorides
C:\Users\Tomek_\Desktop\cc\FGYES1\carboxylic esters
C:\Users\Tomek_\Desktop\cc\FGYES1\alkyl aryl ethers
C:\Users\Tomek_\Desktop\cc\FGYES1\phenols
C:\Users\Tomek_\Desktop\cc\FG

In [9]:
# This cell generates the training-test dataset for a given functional group
# (index of key x). For all groups repeat in loop

x=20
print(keys[x])
import random
import shutil
counter=0;
FGYES='C:\\Users\\Tomek_\\Desktop\\cc\\FGYES1\\'+keys[x]
FGNOT='C:\\Users\\Tomek_\\Desktop\\cc\\FGNOT1\\'+keys[x]

for file in os.listdir(save_dirpath):
    df = pd.read_csv(save_dirpath+file)
    for i in range(len(keys)):
        if df['funcGroups'][x]==1:
            src=save_dirpath+file
            shutil.copy(src, FGYES)
            counter+=1;
        else:
            src=save_dirpath+file
            shutil.copy(src, FGNOT)


YES_count = len([entry for entry in os.listdir(FGYES) if os.path.isfile(os.path.join(FGYES, entry))])
NO_count = len([entry for entry in os.listdir(FGNOT) if os.path.isfile(os.path.join(FGNOT, entry))])
print('divided, we count YES/NO')
print(YES_count)
print(NO_count)
counter=0;
if YES_count<NO_count:
    major=FGNOT
    minor=FGYES
    counter=YES_count
else:
    major=FGYES
    minor=FGNOT;
    counter=NO_count

for file in os.listdir(minor):
    src=minor+"\\"+file
    dest='C:\\Users\\Tomek_\\Desktop\\cc\\FG1\\'+keys[x]
    shutil.copy(src, dest)
    
pliki = [f for f in os.listdir(major)
                 if os.path.isfile(os.path.join(major, f))]  
random.shuffle(pliki)

for i in range(counter):
    src=major+"\\"+pliki[i];
    dest='C:\\Users\\Tomek_\\Desktop\\cc\\FG1\\'+keys[x]
    shutil.copy(src, dest)
    



alkyl aryl ethers
divided, we count YES/NO
39
120
